# Gulfstream walkthrough — yield curves (`zero_rates`)

End-to-end Graph **1** and Graph **2** on real curve data, using the public
`gulfstream` API (`run_single_segmentation`, `refine_regimes`, `plot_regimes`).

| Part | Focus |
|------|--------|
| A | PCA baseline → Graph 1 + Graph 2 |
| B | Kernel PCA → Graph 1 + Graph 2 |
| C | DMD → Graph 1 + Graph 2 |
| D | Search methods: **Binseg** / **BottomUp** (vs PELT) |
| E | Tests: **energy_distance** / **mmd_unbiased** |
| F | Data-driven window: **ESS** heuristic |
| G | Classical hard-label detectors (k-means / HMM) + Graph 2 |
| H | Classical models as soft dimred into kernel_ruptures |
| I | TFT attention embeddings as dimred (+ optional Graph 2) |
| J | Curve / ICA dimred |
| K | Product: uncertainty / CI ribbons / Excel / events / streaming / panel |
| L | Graph 2 retrain scores (`mse_on_diff`, `energy_split`, `mmd_split`, …) |
| — | Comparison (covering + breakpoint F1 vs PCA) |

**Database:** `D:/data/duckdb/ycs_data.duckdb` · **table:** `zero_rates`

Run top-to-bottom. After you finish, tell the agent so outputs can be checked.


## 0. Project setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import duckdb
import pandas as pd
import polars as pl
from plotnine import aes, geom_line, ggplot, labs, theme_bw, facet_wrap, theme

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR.parent
else:
    ROOT = Path(r"D:/Code/gulfstream")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

YCS_DB = Path(r"D:/data/duckdb/ycs_data.duckdb")
OUT_DIR = ROOT / "outputs" / "notebooks" / "ycs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("YCS_DB exists =", YCS_DB.exists())


## 1. Peek at `zero_rates`

Long format: one row per `(date, source)` with tenor columns `Y002p0`, `Y005p0`, …


In [ ]:
con = duckdb.connect(str(YCS_DB), read_only=True)
print("tables:", con.execute("SHOW TABLES").fetchall())
sample = con.execute(
    """
    SELECT date, source, Y002p0, Y005p0, Y010p0, Y030p0
    FROM zero_rates
    WHERE source IN ('USA', 'DEU', 'ITA')
      AND date >= '2015-01-01'
    ORDER BY date, source
    LIMIT 6
    """
).pl()
print(sample)
coverage = con.execute(
    """
    SELECT source, COUNT(*) AS n, MIN(date) AS dmin, MAX(date) AS dmax
    FROM zero_rates
    GROUP BY 1
    ORDER BY 1
    """
).pl()
print(coverage)
con.close()


## 2. Load through the public API

`gulfstream.load_features` reads `config/sources/notebook_ycs.yaml` (USA/DEU/ITA
tenors + FX, then yield-feature engineering).


In [ ]:
from gulfstream import load_features
from gulfstream.common import frames

features_df = load_features(
    ROOT / "config" / "sources" / "notebook_ycs.yaml",
    project_root=ROOT,
)
print("shape:", features_df.shape, "n_features:", frames.n_features(features_df))
print("date range:", features_df["date"].min(), "→", features_df["date"].max())
print("feature sample:", frames.feature_columns(features_df)[:10])
features_df.head(3)


## 3. Explore a few series


In [ ]:
plot_cols = [
    c
    for c in [
        "USA_Y010p0",
        "DEU_Y010p0",
        "ITA_Y010p0",
        "USA_Y002p0_minus_USA_Y010p0",
        "EURUSD",
    ]
    if c in features_df.columns
]
long = (
    features_df.select(["date", *plot_cols])
    .unpivot(index="date", on=plot_cols, variable_name="series", value_name="value")
    .to_pandas()
)
long["date"] = pd.to_datetime(long["date"])

(
    ggplot(long, aes("date", "value", color="series"))
    + geom_line(size=0.4)
    + facet_wrap("~series", scales="free_y", ncol=1)
    + theme_bw()
    + theme(figure_size=(10, 2.2 * len(plot_cols)), legend_position="none")
    + labs(title="Selected yield / FX features", x="", y="")
)


## 4. Shared helpers (public API)

Graph 1 uses `run_single_segmentation`. Graph 2 uses `refine_regimes(..., seed=...)`,
which builds `retrain.regimes_df` from the Graph 1 `SegmentResults`.


In [ ]:
import copy
from IPython.display import Image, display

from gulfstream import (
    plot_regimes,
    refine_regimes,
    regime_intervals,
    run_single_segmentation,
    seed_regimes_from_results,
)
from gulfstream.common import frames, utils
from gulfstream.common.options import (
    ClassicalDetector,
    DetectionBackend,
    SearchMethod,
    StatTest,
)
from gulfstream.metrics.evaluation import (
    adjusted_rand_index,
    breakpoint_precision_recall_f1,
    covering_metric,
)


def load_core_params(img_dir: Path) -> dict:
    """Validated Graph 1 core params with notebook-friendly metrics."""
    params = utils.read_config_yaml(
        str(ROOT / "config" / "graph1" / "default_core.yaml"),
        img_dir=str(img_dir),
        log_dir=str(ROOT / "outputs" / "logs"),
    )
    params["test_num"] = 0
    params["metrics"]["mode"] = "display_and_write"
    params["metrics"]["plot"] = True
    params["metrics"]["dir"] = str(img_dir)
    params["metrics"]["image_dir"] = str(img_dir)
    params["robustness"]["enabled"] = False
    params["stability"]["enabled"] = False
    return params


def with_dimred(params: dict, method: str) -> dict:
    out = copy.deepcopy(params)
    method = method.lower()
    out["algo"]["dimred"] = [method]
    if method == "kpca":
        out["algo"]["kpca_kernel_params"] = [{"kernel": "rbf", "gamma": "median"}]
    elif method == "dmd":
        out["algo"]["dmd_stride"] = [5]
        out["algo"]["dmd_rolling_window"] = [20]
    elif method == "ica":
        out["algo"]["rank"] = [3]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["random_state"] = [42]
        out["algo"]["ica_max_iter"] = [200]
    elif method == "fpca":
        out["algo"]["rank_selection_method"] = ["explained_variance"]
        out["algo"]["threshold"] = [0.9]
        out["algo"]["fpca_smooth_window"] = [3]
    elif method == "nelson_siegel":
        out["algo"]["ns_lambda"] = [0.0609]
    elif method == "dynamic_factor":
        out["algo"]["rank"] = [2]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["factor_order"] = [1]
        out["algo"]["df_maxiter"] = [30]
    elif method != "pca":
        raise ValueError(f"Unsupported dimred: {method}")
    return out


def with_search(params: dict, method, **algo_extras) -> dict:
    out = copy.deepcopy(params)
    out["algo"]["search_method"] = [str(method)]
    for k, v in algo_extras.items():
        out["algo"][k] = v if isinstance(v, list) else [v]
    return out


def with_test(params: dict, choice) -> dict:
    out = copy.deepcopy(params)
    out["test"]["choice"] = [str(choice)]
    return out


def with_ess_window(
    params: dict,
    *,
    ess_fraction: float = 0.25,
    min_window: int = 20,
    max_window: int = 100,
) -> dict:
    out = copy.deepcopy(params)
    out["test"]["window"] = [
        {
            "method": "ess",
            "ess_fraction": ess_fraction,
            "min_window": min_window,
            "max_window": max_window,
        }
    ]
    return out


def with_classical(
    params: dict,
    detector,
    *,
    regimes: int | None = 3,
    min_regime_length: int = 20,
    **algo_extras,
) -> dict:
    """Hard-label classical backend (former --mode legacy)."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.CLASSICAL)]
    out["algo"]["regime_detection_algorithm"] = [str(detector)]
    out["algo"]["dimred"] = ["raw"]
    out["algo"]["feature_map_approx_method"] = ["raw"]
    out["algo"]["post_processing_method"] = ["majority_voting"]
    out["algo"]["min_regime_length"] = [min_regime_length]
    out["algo"]["include_last_regime"] = [True]
    if regimes is not None:
        out["algo"]["regimes"] = [regimes]
    for k, v in algo_extras.items():
        out["algo"][k] = v if isinstance(v, list) else [v]
    return out


def with_model_dimred(params: dict, method, *, regimes: int = 3) -> dict:
    """Use classical models as soft embeddings into kernel_ruptures."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.KERNEL_RUPTURES)]
    out["algo"]["dimred"] = [str(method)]
    out["algo"]["regimes"] = [regimes]
    return out


def with_tft(
    params: dict,
    *,
    rank: int = 8,
    encoder_length: int = 20,
    prediction_length: int = 5,
    max_epochs: int = 1,
    batch_size: int = 16,
    mode: str = "multivariate",
) -> dict:
    """TFT attention embeddings → kernel_ruptures (smoke-friendly defaults)."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.KERNEL_RUPTURES)]
    out["algo"]["dimred"] = ["tft"]
    out["algo"]["rank"] = [rank]
    out["algo"]["rank_selection_method"] = ["user_specified"]
    out["algo"]["tft_encoder_length"] = [encoder_length]
    out["algo"]["tft_prediction_length"] = [prediction_length]
    out["algo"]["tft_max_epochs"] = [max_epochs]
    out["algo"]["tft_batch_size"] = [batch_size]
    out["algo"]["tft_mode"] = [mode]
    # Keep the ruptures grid small — TFT itself is the expensive step.
    out["algo"]["num_features"] = [30]
    out["algo"]["depth"] = [1]
    return out


def summarize(res, label: str, df: pl.DataFrame) -> None:
    dates = frames.dates_series(df).to_list()
    print(f"[{label}] kept={res.bkpts}  invalid={res.invalid_bkpts}")
    for b in res.bkpts:
        print(f"  bkpt {b} → {dates[b]}")


def run_g1(
    df: pl.DataFrame,
    params: dict,
    label: str,
    plot_vars: list[str],
    *,
    return_fig: bool = False,
):
    """Graph 1 via public single-pass API.

    Returns ``proc`` by default. Pass ``return_fig=True`` to also get the plotnine
    ggplot (without auto-displaying it) for explicit notebook inspection.
    """
    print(
        f"=== Graph 1 · {label} · backend={params['algo'].get('detection_backend')} "
        f"dimred={params['algo']['dimred']} "
        f"detector={params['algo'].get('regime_detection_algorithm')} "
        f"search={params['algo'].get('search_method')} "
        f"test={params['test'].get('choice')} ==="
    )
    proc = run_single_segmentation(df, params)
    summarize(proc, label, df)
    metrics = params.get("metrics", {})
    img_dir = metrics.get("image_dir") or metrics.get("dir")
    if return_fig:
        plot_mode = "write" if img_dir else "display"
        plot_emit = bool(img_dir)
    else:
        plot_mode = "display_and_write" if img_dir else "display"
        plot_emit = True
    fig = plot_regimes(
        df,
        proc,
        variables=plot_vars[:2],
        title=f"Graph 1 · {label}",
        mode=plot_mode,
        img_dir=str(img_dir) if img_dir else None,
        emit=plot_emit,
    )
    if return_fig:
        return proc, fig
    return proc


def run_g2(
    df: pl.DataFrame,
    params: dict,
    seed_res,
    out_dir: Path,
    label: str,
    plot_vars: list[str],
    *,
    max_iter: int = 3,
    threshold: float = 1e-6,
    score_method: str = "mse_to_mean",
    score: dict | None = None,
    return_figs: bool = False,
):
    """Graph 2 via refine_regimes, seeded from a Graph 1 SegmentResults.

    Returns ``out_dir`` by default. Pass ``return_figs=True`` to also get
    ``refined`` and a ``figs`` dict mapping string keys to plotnine ggplots
    (``retrain_iteration_*`` heatmaps + ``regime``).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    g2 = copy.deepcopy(params)
    g2["metrics"]["dir"] = str(out_dir)
    g2["metrics"]["image_dir"] = str(out_dir)
    g2["metrics"]["mode"] = "display_and_write"
    g2["metrics"]["plot"] = True
    g2["retrain"] = {
        "interactive": False,
        "features": ["__auto__"],
        "num_worst_features": min(5, frames.n_features(df)),
        "threshold": threshold,
        "max_iter": max_iter,
        "score_method": score_method,
        "score": dict(score or {}),
        "regimes_df": None,
    }
    print(f"=== Graph 2 · {label} · score_method={score_method} · seeding from Graph 1 ===")
    print(seed_regimes_from_results(df, seed_res).to_dicts())
    refined = refine_regimes(df, g2, seed=seed_res)
    figs: dict = {}
    if refined is not None:
        summarize(refined, f"{label} Graph 2", df)
        if return_figs:
            figs.update(getattr(refined, "plots", None) or {})
            figs["regime"] = plot_regimes(
                df,
                refined,
                variables=plot_vars[:2],
                title=f"Graph 2 · {label}",
                mode="write",
                img_dir=str(out_dir),
                emit=False,
            )
        else:
            plot_regimes(
                df,
                refined,
                variables=plot_vars[:2],
                title=f"Graph 2 · {label}",
                mode="display",
            )
    if not return_figs:
        pngs = sorted(out_dir.rglob("retrain_iteration_*.png"))[:6]
        print(f"Graph 2 artifacts under {out_dir}")
        for p in pngs:
            print(" ", p.relative_to(out_dir))
            try:
                display(Image(filename=str(p)))
            except Exception as exc:
                print("  (could not display)", exc)
    else:
        print(f"Graph 2 artifacts under {out_dir} ({len(figs)} plotnine figure(s))")
        print(" fig keys:", sorted(figs))
    if return_figs:
        return out_dir, refined, figs
    return out_dir

print("Helpers ready: run_g1/g2, with_dimred/search/test/ess/classical/model_dimred/tft")
print("Enums:", list(DetectionBackend), list(ClassicalDetector)[:4], "...")


---
# Part A — PCA (baseline)

Default core: **PCA → RFF → PELT → MMD**, then Graph 2 seeded from that run.


## A.1 Graph 1 (PCA + PELT)


In [ ]:
params_pca = with_dimred(load_core_params(OUT_DIR / "pca"), "pca")
params_pca["metrics"]["features_to_plot"] = plot_cols[:3]
proc_pca, fig_pca = run_g1(features_df, params_pca, "PCA", plot_cols, return_fig=True)
regime_intervals(proc_pca, frames.dates_series(features_df).to_list())
fig_pca


## A.2 Graph 2 (seeded from PCA)

Auto-retrain loop: feature×regime **score heatmap** (default `mse_to_mean`).
Part L swaps `retrain.score_method` for curve-friendly alternatives.


In [ ]:
g2_pca_dir, proc_pca_g2, g2_pca_figs = run_g2(
    features_df, params_pca, proc_pca, OUT_DIR / "pca" / "graph2", "PCA", plot_cols, max_iter=3,
    return_figs=True,
)
g2_pca_figs["regime"]


---
# Part B — Kernel PCA


## B.1 Graph 1 (kPCA)


In [ ]:
params_kpca = with_dimred(load_core_params(OUT_DIR / "kpca"), "kpca")
params_kpca["metrics"]["features_to_plot"] = plot_cols[:3]
proc_kpca = run_g1(features_df, params_kpca, "kPCA", plot_cols)


## B.2 Graph 2 (seeded from kPCA)


In [ ]:
g2_kpca_dir = run_g2(
    features_df, params_kpca, proc_kpca, OUT_DIR / "kpca" / "graph2", "kPCA", plot_cols, max_iter=3
)


---
# Part C — DMD


## C.1 Graph 1 (DMD)


In [ ]:
params_dmd = with_dimred(load_core_params(OUT_DIR / "dmd"), "dmd")
params_dmd["metrics"]["features_to_plot"] = plot_cols[:3]
proc_dmd = run_g1(features_df, params_dmd, "DMD", plot_cols)


## C.2 Graph 2 (seeded from DMD)


In [ ]:
g2_dmd_dir = run_g2(
    features_df, params_dmd, proc_dmd, OUT_DIR / "dmd" / "graph2", "DMD", plot_cols, max_iter=3
)


---
# Part D — Search methods (Binseg / BottomUp / WBS / BOCPD)

Same PCA embedding and MMD test as Part A; only `algo.search_method` changes.
Compare candidate-generation strategies (including **WBS** and **BOCPD**) against
the PELT baseline.


## D.1 Binseg


In [ ]:
params_binseg = with_search(
    with_dimred(load_core_params(OUT_DIR / "binseg"), "pca"),
    SearchMethod.BINSEG,
)
params_binseg["metrics"]["features_to_plot"] = plot_cols[:3]
proc_binseg = run_g1(features_df, params_binseg, "Binseg", plot_cols)


## D.2 BottomUp


In [ ]:
params_bottomup = with_search(
    with_dimred(load_core_params(OUT_DIR / "bottomup"), "pca"),
    SearchMethod.BOTTOMUP,
)
params_bottomup["metrics"]["features_to_plot"] = plot_cols[:3]
proc_bottomup = run_g1(features_df, params_bottomup, "BottomUp", plot_cols)


## D.3 Wild Binary Segmentation (WBS)


In [ ]:
params_wbs = with_search(
    with_dimred(load_core_params(OUT_DIR / "wbs"), "pca"),
    SearchMethod.WBS,
    wbs_n_intervals=200,
    random_state=42,
)
params_wbs["metrics"]["features_to_plot"] = plot_cols[:3]
proc_wbs = run_g1(features_df, params_wbs, "WBS", plot_cols)


## D.4 Bayesian Online Changepoint Detection (BOCPD)


In [ ]:
params_bocpd = with_search(
    with_dimred(load_core_params(OUT_DIR / "bocpd"), "pca"),
    SearchMethod.BOCPD,
    bocpd_hazard=0.01,
    bocpd_threshold=0.4,
    bocpd_max_run=200,
)
params_bocpd["metrics"]["features_to_plot"] = plot_cols[:3]
proc_bocpd = run_g1(features_df, params_bocpd, "BOCPD", plot_cols)


---
# Part E — Statistical tests

Same PCA + PELT search as Part A; swap `test.choice` across energy distance,
unbiased / linear-time MMD, Hotelling T², multivariate CUSUM, and KS on PCA scores.


## E.1 Energy distance


In [ ]:
params_energy = with_test(
    with_dimred(load_core_params(OUT_DIR / "energy"), "pca"),
    StatTest.ENERGY_DISTANCE,
)
params_energy["metrics"]["features_to_plot"] = plot_cols[:3]
proc_energy = run_g1(features_df, params_energy, "energy_distance", plot_cols)


## E.2 Unbiased MMD


In [ ]:
params_mmd_u = with_test(
    with_dimred(load_core_params(OUT_DIR / "mmd_unbiased"), "pca"),
    StatTest.MMD_UNBIASED,
)
params_mmd_u["metrics"]["features_to_plot"] = plot_cols[:3]
proc_mmd_u = run_g1(features_df, params_mmd_u, "mmd_unbiased", plot_cols)


## E.3 Linear-time MMD


In [ ]:
params_mmd_linear = with_test(
    with_dimred(load_core_params(OUT_DIR / "mmd_linear"), "pca"),
    StatTest.MMD_LINEAR,
)
params_mmd_linear["metrics"]["features_to_plot"] = plot_cols[:3]
proc_mmd_lin = run_g1(features_df, params_mmd_linear, "mmd_linear", plot_cols)


## E.4 Hotelling T²


In [ ]:
params_hotelling_t2 = with_test(
    with_dimred(load_core_params(OUT_DIR / "hotelling_t2"), "pca"),
    StatTest.HOTELLING_T2,
)
params_hotelling_t2["metrics"]["features_to_plot"] = plot_cols[:3]
proc_hotelling = run_g1(features_df, params_hotelling_t2, "hotelling_t2", plot_cols)


## E.5 Multivariate CUSUM


In [ ]:
params_multivariate_cusum = with_test(
    with_dimred(load_core_params(OUT_DIR / "multivariate_cusum"), "pca"),
    StatTest.MULTIVARIATE_CUSUM,
)
params_multivariate_cusum["metrics"]["features_to_plot"] = plot_cols[:3]
proc_mcusum = run_g1(features_df, params_multivariate_cusum, "multivariate_cusum", plot_cols)


## E.6 KS on PCA scores


In [ ]:
params_ks_pca = with_test(
    with_dimred(load_core_params(OUT_DIR / "ks_pca"), "pca"),
    StatTest.KS_PCA,
)
params_ks_pca["metrics"]["features_to_plot"] = plot_cols[:3]
proc_ks_pca = run_g1(features_df, params_ks_pca, "ks_pca", plot_cols)


---
# Part F — ESS window

Replace the fixed MMD window with the **effective-sample-size** heuristic
(`test.window.method: ess`). Search and test stay at PCA + PELT + MMD.


In [ ]:
params_ess = with_ess_window(
    with_dimred(load_core_params(OUT_DIR / "ess"), "pca"),
    ess_fraction=0.25,
    min_window=20,
    max_window=100,
)
params_ess["metrics"]["features_to_plot"] = plot_cols[:3]
proc_ess = run_g1(features_df, params_ess, "ESS window", plot_cols)


---
# Part G — Classical hard-label detectors

Former **legacy** mode is now Graph 1 with `algo.detection_backend: classical`.
Detectors assign labels → breakpoints (no RFF / PELT / MMD). Same public API:
`run_single_segmentation` / `refine_regimes`.


## G.1 k-means (classical)


In [ ]:
params_ckmeans = with_classical(
    load_core_params(OUT_DIR / "classical_kmeans"),
    ClassicalDetector.KMEANS,
    regimes=3,
    random_state=42,
)
params_ckmeans["metrics"]["features_to_plot"] = plot_cols[:3]
proc_ckmeans = run_g1(features_df, params_ckmeans, "classical kmeans", plot_cols)


## G.2 HMM (classical)


In [ ]:
params_chmm = with_classical(
    load_core_params(OUT_DIR / "classical_hmm"),
    ClassicalDetector.HMM,
    regimes=3,
    hmm_emissions="gaussian",
    hmm_n_iter=50,
)
params_chmm["metrics"]["features_to_plot"] = plot_cols[:3]
proc_chmm = run_g1(features_df, params_chmm, "classical HMM", plot_cols)


## G.3 Jump model (classical)


In [ ]:
params_cjump = with_classical(
    load_core_params(OUT_DIR / "classical_jump_model"),
    ClassicalDetector.JUMP_MODEL,
    regimes=3,
    jump_penalty=5.0,
    jump_max_iter=20,
)
params_cjump["metrics"]["features_to_plot"] = plot_cols[:3]
proc_cjump = run_g1(features_df, params_cjump, "classical jump_model", plot_cols)


## G.4 Sticky HDP-HMM (classical)


In [ ]:
params_chdp = with_classical(
    load_core_params(OUT_DIR / "classical_sticky_hdp_hmm"),
    ClassicalDetector.STICKY_HDP_HMM,
    regimes=3,
)
params_chdp["metrics"]["features_to_plot"] = plot_cols[:3]
proc_chdp = run_g1(features_df, params_chdp, "classical sticky_hdp_hmm", plot_cols)


## G.5 GARCH volatility regimes (classical)


In [ ]:
params_cgarch = with_classical(
    load_core_params(OUT_DIR / "classical_garch"),
    ClassicalDetector.GARCH,
    regimes=2,
    garch_p=1,
    garch_q=1,
)
params_cgarch["metrics"]["features_to_plot"] = plot_cols[:3]
proc_cgarch = run_g1(features_df, params_cgarch, "classical garch", plot_cols)


## G.6 Graph 2 seeded from classical k-means


In [ ]:
g2_ckmeans_dir = run_g2(
    features_df,
    params_ckmeans,
    proc_ckmeans,
    OUT_DIR / "classical_kmeans" / "graph2",
    "classical kmeans",
    plot_cols,
    max_iter=2,
)


---
# Part H — Classical models as soft dimred

Same algorithm families as Part G, but as **embeddings** into the default
`kernel_ruptures` stack (`algo.dimred: [kmeans|hmm]` → RFF → PELT → MMD).
This is how Graph 1 used “legacy” models without leaving the ruptures path.


## H.1 k-means dimred → kernel ruptures


In [ ]:
params_kmeans_dim = with_model_dimred(
    load_core_params(OUT_DIR / "kmeans_dimred"),
    "kmeans",
    regimes=3,
)
params_kmeans_dim["metrics"]["features_to_plot"] = plot_cols[:3]
proc_kmeans_dim = run_g1(features_df, params_kmeans_dim, "kmeans dimred", plot_cols)


## H.2 HMM dimred → kernel ruptures


In [ ]:
params_hmm_dim = with_model_dimred(
    load_core_params(OUT_DIR / "hmm_dimred"),
    "hmm",
    regimes=3,
)
params_hmm_dim["metrics"]["features_to_plot"] = plot_cols[:3]
proc_hmm_dim = run_g1(features_df, params_hmm_dim, "HMM dimred", plot_cols)


---
# Part I — TFT dimred (Temporal Fusion Transformer)

Train a short TFT and use its **attention vectors** as the Graph 1 embedding
(`algo.dimred: [tft]` → RFF → PELT → MMD). Needs `torch`, `lightning`, and
`pytorch-forecasting`.

Notebook defaults are smoke settings (1 epoch, encoder=20). Prefer
**multivariate** mode on wide panels; univariate melts every feature into its
own series and is much slower.

Feature columns with `.` are renamed inside `gulfstream.detectors.tft` before
training (no notebook-side sanitization needed).


## I.1 Graph 1 (TFT attention embeddings)


In [ ]:
# Skip cleanly if the optional TFT stack is missing.
try:
    import torch  # noqa: F401
    import lightning  # noqa: F401
    import pytorch_forecasting  # noqa: F401
    _TFT_OK = True
except ImportError as exc:
    _TFT_OK = False
    print("TFT stack unavailable — skipping Part I:", exc)

if _TFT_OK:
    params_tft = with_tft(load_core_params(OUT_DIR / "tft"))
    params_tft["metrics"]["features_to_plot"] = plot_cols[:3]
    print(
        "TFT smoke:",
        f"n={features_df.height}",
        f"enc={params_tft['algo']['tft_encoder_length']}",
        f"epochs={params_tft['algo']['tft_max_epochs']}",
        f"mode={params_tft['algo']['tft_mode']}",
    )
    proc_tft = run_g1(features_df, params_tft, "TFT dimred", plot_cols)
else:
    proc_tft = proc_pca  # placeholder so the comparison cell still runs


## I.2 Graph 2 seeded from TFT (optional, still expensive)


In [ ]:
if _TFT_OK:
    g2_tft_dir = run_g2(
        features_df,
        params_tft,
        proc_tft,
        OUT_DIR / "tft" / "graph2",
        "TFT",
        plot_cols,
        max_iter=1,
    )
else:
    print("Skipping TFT Graph 2")


---
# Part J — Curve / ICA dimred

Roadmap dimred: **ICA**, **functional PCA**, **Nelson–Siegel** (curve-friendly for
zero rates), and **dynamic factors**. Same kernel_ruptures path as Part A.


## J.1 ICA


In [ ]:
params_ica = with_dimred(load_core_params(OUT_DIR / "ica"), "ica")
params_ica["metrics"]["features_to_plot"] = plot_cols[:3]
proc_ica = run_g1(features_df, params_ica, "ICA dimred", plot_cols)


## J.2 Functional PCA


In [ ]:
params_fpca = with_dimred(load_core_params(OUT_DIR / "fpca"), "fpca")
params_fpca["metrics"]["features_to_plot"] = plot_cols[:3]
proc_fpca = run_g1(features_df, params_fpca, "FPCA dimred", plot_cols)


## J.3 Nelson–Siegel


In [ ]:
params_ns = with_dimred(load_core_params(OUT_DIR / "nelson_siegel"), "nelson_siegel")
params_ns["metrics"]["features_to_plot"] = plot_cols[:3]
proc_ns = run_g1(features_df, params_ns, "Nelson–Siegel dimred", plot_cols)


## J.4 Dynamic factor


In [ ]:
params_dfactor = with_dimred(load_core_params(OUT_DIR / "dynamic_factor"), "dynamic_factor")
params_dfactor["metrics"]["features_to_plot"] = plot_cols[:3]
proc_dfactor = run_g1(features_df, params_dfactor, "dynamic_factor dimred", plot_cols)


---
# Part K — Product features (uncertainty, export, events, streaming, panel)

Walk through the product knobs added for dashboards and ops:

1. **Uncertainty bands** → `SegmentResults.bkpt_ci`, shaded as **CI ribbons** on regime plots
2. **Excel export** + **NDJSON event stream** (optional `path` / `dir`+`filename`)
3. **Streaming** Graph 1 (expanding window)
4. **Panel joint** breakpoints across curve sources (USA / DEU / ITA)


## K.1 Uncertainty bands + CI ribbon overlays

Bootstrap ensembles attach calibrated `(lo, hi)` index bands. Regime plots shade them when
`metrics.plot_ci_ribbons` is true (default).


In [ ]:
from gulfstream.metrics import uncertainty as uncertainty_mod

product_dir = OUT_DIR / "product"
product_dir.mkdir(parents=True, exist_ok=True)

params_unc = with_dimred(load_core_params(product_dir / "uncertainty"), "pca")
params_unc["metrics"]["features_to_plot"] = plot_cols
params_unc["metrics"]["plot_ci_ribbons"] = True
params_unc["uncertainty"] = {
    "enabled": True,
    "sources": ["bootstrap"],
    "level": 0.9,
    "match_tolerance": 5,
    "n_bootstrap": 4,
    "bootstrap_block": 25,
    "random_state": 42,
}

proc_unc = run_g1(features_df, params_unc, "PCA + uncertainty", plot_cols)
# Single-pass does not attach CI; calibrate bands explicitly (same as produce_all_metrics).
proc_unc = uncertainty_mod.evaluate_uncertainty(
    features_df,
    {**params_unc, "_pipeline_params": params_unc},
    proc_unc,
)
print("bkpt_ci:", proc_unc.bkpt_ci)
fig_graph_1_pca_ci_ribbons = plot_regimes(
    features_df,
    proc_unc,
    variables=plot_cols[:2],
    title="Graph 1 · PCA + CI ribbons",
    mode="display",
    emit=False,
)
fig_graph_1_pca_ci_ribbons


## K.2 Excel export + dashboard NDJSON event stream

Optional location via `export.excel.path` **or** `dir` + `filename` (defaults under
`metrics.dir`). Events are one JSON object per line (`run_started`, `breakpoint_confirmed`,
`run_complete`, …).

Example YAML: `config/graph1/graph1_export_events.yaml`.


In [ ]:
import json

from gulfstream.metrics.writers import export_breakpoint_excel
from gulfstream.ops.events import emit_run_events
import pandas as pd

export_dir = product_dir / "export_events"
export_dir.mkdir(parents=True, exist_ok=True)

params_export = copy.deepcopy(params_unc)
params_export["metrics"]["dir"] = str(export_dir)
params_export["metrics"]["image_dir"] = str(export_dir)
params_export["export"] = {
    "excel": {
        "enabled": True,
        "dir": str(export_dir),
        "filename": "bkpt_export.xlsx",
    }
}
params_export["events"] = {
    "enabled": True,
    "dir": str(export_dir),
    "filename": "events.ndjson",
    "append": False,
}

xlsx_path = export_breakpoint_excel(
    proc_unc,
    params_export,
    dates=frames.dates_series(features_df).to_list(),
)
ndjson_path = emit_run_events(params_export, proc_unc)
print("Excel:", xlsx_path)
print("Events:", ndjson_path)

if ndjson_path:
    for line in Path(ndjson_path).read_text(encoding="utf-8").strip().splitlines():
        evt = json.loads(line)
        print(f"  {evt['event']}:", {k: evt[k] for k in evt if k not in ("ts", "event")})

if xlsx_path:
    for sheet in ("Breakpoints", "CI", "PanelSupport", "Meta"):
        print(f"\n--- {sheet} ---")
        display(pd.read_excel(xlsx_path, sheet_name=sheet))


## K.3 Streaming Graph 1 (expanding)

Advance an expanding window and lock confirmed breaks. Use `detect_regimes` with
`streaming.enabled`, or step with `detect_regimes_incremental`.


In [ ]:
from gulfstream import detect_regimes_incremental

stream_dir = product_dir / "streaming"
stream_dir.mkdir(parents=True, exist_ok=True)

params_stream = with_dimred(load_core_params(stream_dir), "pca")
params_stream["metrics"]["plot"] = False  # keep the incremental loop snappy
params_stream["streaming"] = {
    "enabled": True,
    "mode": "expanding",
    "step": 80,
    "min_history": 200,
    "lock_prefix": True,
    "match_tolerance": 5,
}

state = None
proc_stream = None
for step_i in range(3):
    proc_stream, state = detect_regimes_incremental(features_df, params_stream, state)
    print(
        f"step={step_i} last_t={state.last_t if state is not None else '?'} "
        f"bkpts={proc_stream.bkpts}"
    )

summarize(proc_stream, "streaming (last step)", features_df)
fig_streaming_graph_1_last_step = plot_regimes(
    features_df,
    proc_stream,
    variables=plot_cols[:2],
    title="Streaming Graph 1 (last step)",
    mode="display",
    emit=False,
)
fig_streaming_graph_1_last_step


## K.4 Panel joint breakpoints

Segment each curve **source** separately, then take a majority / intersection / union
consensus. Support fractions land in `panel_support` (also exported in the Excel workbook).


In [ ]:
from gulfstream import detect_regimes_panel

panel_dir = product_dir / "panel"
panel_dir.mkdir(parents=True, exist_ok=True)

params_panel = with_dimred(load_core_params(panel_dir), "pca")
params_panel["metrics"]["plot"] = False
params_panel["panel"] = {
    "enabled": True,
    "groupby": "source",  # USA_ / DEU_ / ITA_ prefixes
    "combine": "majority",
    "min_group_frac": 0.5,
    "match_tolerance": 5,
}

proc_panel = detect_regimes_panel(features_df, params_panel)
summarize(proc_panel, "panel majority", features_df)
print("panel_support:", proc_panel.panel_support)
fig_panel_joint_source_majority = plot_regimes(
    features_df,
    proc_panel,
    variables=plot_cols[:2],
    title="Panel joint (source majority)",
    mode="display",
    emit=False,
)
fig_panel_joint_source_majority


---
# Part L — Graph 2 retrain score methods

Graph 2 picks the next slice from a **feature × regime** score matrix
(`retrain.score_method`). Default `mse_to_mean` is the legacy L2 heatmap.
Alternatives are better for drifting levels, panel co-movement, or remaining
changepoint evidence. **`threshold` is in the chosen score’s units** — retune
when switching methods.

| Method | Idea |
|--------|------|
| `mse_to_mean` | L2 to regime mean (default) |
| `mad_to_median` | Robust L1 twin |
| `mse_on_diff` | MSE on first differences |
| `factor_residual` | Within-regime PCA residual (panel fit) |
| `hotelling_within` / `cusum_intensity` | Detection-aligned leftovers |
| `energy_split` / `mmd_split` | Best mid-window two-sample split |


## L.1 Compare score matrices on the PCA Graph 1 seed

No Graph 2 run yet — just ask each scorer which (feature, regime) it would refine first.


In [ ]:
import numpy as np

from gulfstream.metrics.regime_scores import known_score_methods, score_feature_regime

score_dir = OUT_DIR / "graph2_scores"
score_dir.mkdir(parents=True, exist_ok=True)

# Keep the comparison readable: score on the plot columns only.
feat_for_scores = [c for c in plot_cols if c in frames.feature_columns(features_df)]
score_df = frames.select_features(features_df, feat_for_scores)
bkpts_seed = list(proc_pca.bkpts)
feat_names = frames.feature_columns(score_df)

split_kwargs = {"n_splits": 5, "min_side": 10, "max_rows": 80}
specs = [
    ("mse_to_mean", {}),
    ("mad_to_median", {}),
    ("mse_on_diff", {"diff_order": 1}),
    ("factor_residual", {"n_components": 1}),
    ("hotelling_within", {}),
    ("cusum_intensity", {}),
    ("energy_split", split_kwargs),
    ("mmd_split", {**split_kwargs, "mmd_estimator": "linear"}),
]

rows = []
for method, kwargs in specs:
    mat = score_feature_regime(score_df, bkpts_seed, method, **kwargs)
    fi, ri = np.unravel_index(int(np.argmax(mat)), mat.shape)
    rows.append(
        {
            "score_method": method,
            "worst_feature": feat_names[fi],
            "worst_regime": int(ri),
            "max_score": float(mat[fi, ri]),
            "n_feat": mat.shape[0],
            "n_regimes": mat.shape[1],
        }
    )
    print(f"{method:18s} → {feat_names[fi]} @ regime {ri}  (max={mat[fi, ri]:.4g})")

print("Available methods:", known_score_methods())
score_pick_table = pl.DataFrame(rows)
score_pick_table


## L.2 Graph 2 with `mse_on_diff` and `factor_residual`

Differenced MSE suits drifting yield levels; factor residual targets tenors that
co-move less within a regime. Smoke `max_iter=2`.


In [ ]:
params_g2_scores = with_dimred(load_core_params(score_dir), "pca")
params_g2_scores["metrics"]["features_to_plot"] = plot_cols
params_g2_scores["metrics"]["plot"] = False  # heatmaps still written by Graph 2 loop

g2_diff_dir = run_g2(
    features_df,
    params_g2_scores,
    proc_pca,
    score_dir / "mse_on_diff",
    "PCA · mse_on_diff",
    plot_cols,
    max_iter=2,
    threshold=1e-6,
    score_method="mse_on_diff",
    score={"diff_order": 1},
)

g2_factor_dir = run_g2(
    features_df,
    params_g2_scores,
    proc_pca,
    score_dir / "factor_residual",
    "PCA · factor_residual",
    plot_cols,
    max_iter=2,
    threshold=1e-6,
    score_method="factor_residual",
    score={"n_components": 1},
)
print("diff heatmaps:", list(g2_diff_dir.rglob("retrain_iteration_*.png"))[:3])
print("factor heatmaps:", list(g2_factor_dir.rglob("retrain_iteration_*.png"))[:3])


## L.3 Graph 2 with `energy_split` (and optional `mmd_split`)

Kernel-aligned scorers: max mid-window energy / MMD inside each regime. Keep
`n_splits` / `max_rows` small for notebook runtime.


In [ ]:
g2_energy_dir = run_g2(
    features_df,
    params_g2_scores,
    proc_pca,
    score_dir / "energy_split",
    "PCA · energy_split",
    plot_cols,
    max_iter=2,
    threshold=1e-6,
    score_method="energy_split",
    score={"n_splits": 5, "min_side": 10, "max_rows": 80},
)

g2_mmd_dir = run_g2(
    features_df,
    params_g2_scores,
    proc_pca,
    score_dir / "mmd_split",
    "PCA · mmd_split",
    plot_cols,
    max_iter=2,
    threshold=1e-6,
    score_method="mmd_split",
    score={
        "n_splits": 5,
        "min_side": 10,
        "max_rows": 80,
        "mmd_estimator": "linear",
    },
)
print("energy:", g2_energy_dir)
print("mmd:", g2_mmd_dir)


---
# Comparison

Covering, **adjusted Rand index**, and breakpoint F1 (tolerance = 10 days) against
the **PCA / PELT / MMD** baseline from Part A. Part K product runs and Part L
score picks are included when those cells were executed.


In [ ]:
dates = frames.dates_series(features_df).to_list()
baseline = proc_pca
n = features_df.height

def row(label: str, res) -> dict:
    f1 = breakpoint_precision_recall_f1(baseline.bkpts, res.bkpts, tolerance=10)
    return {
        "run": label,
        "n_bkpts": len(res.bkpts),
        "bkpts": res.bkpts,
        "dates": [str(dates[b]) for b in res.bkpts],
        "covering_vs_pca": covering_metric(baseline.bkpts, res.bkpts, n),
        "ari_vs_pca": adjusted_rand_index(baseline.bkpts, res.bkpts, n),
        "f1_vs_pca": f1["f1"],
        "precision_vs_pca": f1["precision"],
        "recall_vs_pca": f1["recall"],
    }

rows = [
    row("A pca/pelt/mmd", proc_pca),
    row("B kpca", proc_kpca),
    row("C dmd", proc_dmd),
    row("D binseg", proc_binseg),
    row("D bottomup", proc_bottomup),
    row("D wbs", proc_wbs),
    row("D bocpd", proc_bocpd),
    row("E energy", proc_energy),
    row("E mmd_unbiased", proc_mmd_u),
    row("E mmd_linear", proc_mmd_lin),
    row("E hotelling_t2", proc_hotelling),
    row("E multivariate_cusum", proc_mcusum),
    row("E ks_pca", proc_ks_pca),
    row("F ess window", proc_ess),
    row("G classical kmeans", proc_ckmeans),
    row("G classical hmm", proc_chmm),
    row("G classical jump_model", proc_cjump),
    row("G classical sticky_hdp_hmm", proc_chdp),
    row("G classical garch", proc_cgarch),
    row("H kmeans dimred", proc_kmeans_dim),
    row("H hmm dimred", proc_hmm_dim),
    row("I tft dimred", proc_tft),
    row("J ica", proc_ica),
    row("J fpca", proc_fpca),
    row("J nelson_siegel", proc_ns),
    row("J dynamic_factor", proc_dfactor),
]
if "proc_unc" in globals():
    rows.append(row("K uncertainty", proc_unc))
if "proc_panel" in globals():
    rows.append(row("K panel", proc_panel))

summary = pl.DataFrame(rows)
if "score_pick_table" in globals():
    print("Part L score picks (from L.1):")
    display(score_pick_table)
summary


## CLI equivalents

```bash
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/default_core.yaml \
  --source-config config/sources/notebook_ycs.yaml

uv run python -m gulfstream.cli --mode graph2 \
  --config config/graph2/full_graph2.yaml \
  --source-config config/sources/notebook_ycs.yaml

# Search: WBS / BOCPD
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_wbs_search.yaml \
  --source-config config/sources/notebook_ycs.yaml
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_bocpd_search.yaml \
  --source-config config/sources/notebook_ycs.yaml

# Classical hard-label (incl. jump_model / sticky_hdp_hmm / garch)
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/classical_kmeans.yaml \
  --source-config config/sources/notebook_ycs.yaml
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/classical_jump_model.yaml \
  --source-config config/sources/notebook_ycs.yaml

# Soft / curve dimred
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_kmeans_dimred.yaml \
  --source-config config/sources/notebook_ycs.yaml
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_nelson_siegel_dimred.yaml \
  --source-config config/sources/notebook_ycs.yaml
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_tft_dimred.yaml \
  --source-config config/sources/notebook_ycs.yaml

# Product: uncertainty + Excel + NDJSON events / streaming / panel
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_export_events.yaml \
  --source-config config/sources/notebook_ycs.yaml
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_streaming_expanding.yaml \
  --source-config config/sources/notebook_ycs.yaml
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_panel_joint.yaml \
  --source-config config/sources/notebook_ycs.yaml

# Graph 2 score methods (retune retrain.threshold per method)
uv run python -m gulfstream.cli --mode graph2 \
  --config config/graph2/graph2_score_diff.yaml \
  --source-config config/sources/notebook_ycs.yaml
uv run python -m gulfstream.cli --mode graph2 \
  --config config/graph2/graph2_score_factor.yaml \
  --source-config config/sources/notebook_ycs.yaml
uv run python -m gulfstream.cli --mode graph2 \
  --config config/graph2/graph2_score_energy.yaml \
  --source-config config/sources/notebook_ycs.yaml
uv run python -m gulfstream.cli --mode graph2 \
  --config config/graph2/graph2_score_mmd.yaml \
  --source-config config/sources/notebook_ycs.yaml
```

Set `algo.search_method`, `test.choice`, `test.window`,
`algo.detection_backend: [classical]`, or `algo.dimred` to match Parts D–J.


## What to try next

- Change the USA/DEU/ITA tenor set or date window in `config/sources/notebook_ycs.yaml`.
- Combine knobs (e.g. WBS + `hotelling_t2` + ESS).
- Classical `jump_model` / `sticky_hdp_hmm` / `garch`.
- Curve dimred: `nelson_siegel`, `fpca`, `dynamic_factor`, or `ica`.
- TFT: raise `tft_max_epochs`, or use `config/graph1/graph1_tft_dimred.yaml`.
- Product: `config/graph1/graph1_export_events.yaml`, streaming / panel / uncertainty YAMLs.
- Graph 2 scores: `config/graph2/graph2_score_{diff,factor,energy,mmd}.yaml` — retune `retrain.threshold`.
- Raise Graph 2 `max_iter` or lower `threshold`.
